In [0]:
# Load Tables

from pyspark.sql.functions import col

proc_df = spark.table("medical_project.gold.fact_procedures")
date_df = spark.table("medical_project.gold.dim_date")

In [0]:
# Join Date Dimension

df = proc_df.join(
    date_df,
    proc_df.procedure_date == date_df.date,
    "left"
)
# Ensure Correct Data Types

df = df.withColumn("year", col("year").cast("int")) \
       .withColumn("month", col("month").cast("int"))
# Create Half-Year Column

from pyspark.sql.functions import when

df = df.withColumn(
    "half_year",
    when(col("month") <= 6, 1).otherwise(2)   # keep numeric (1,2)
)

In [0]:
# Build Cube

from pyspark.sql.functions import count, sum

proc_cube = df.cube(
    "year",
    "half_year",
    "month",
    "procedure_description"
).agg(
    count("*").alias("procedure_count"),
    sum("base_cost").alias("total_cost")
)

In [0]:
# Save Cube

proc_cube.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.gold.procedure_cube")